In [2]:
import json
from pathlib import Path

base = Path(r"C:\Users\Muazb\project\data\raw\pallets__flask")
print(base.exists(), len(list((base / "pullRequests").glob("page_*.json"))))

def load(kind):
    items = []
    for f in sorted((base / kind).glob("page_*.json")):
        items += json.loads(f.read_text(encoding="utf-8"))["nodes"]
    return items

issues = load("issues")
prs = load("pullRequests")

linked = [p for p in prs if p["closingIssuesReferences"]["nodes"]]
merged = [p for p in prs if p["mergedAt"]]
linked_merged = [p for p in linked if p["mergedAt"]]

print(f"issues: {len(issues)}")
print(f"PRs: {len(prs)} (merged: {len(merged)})")
print(f"PRs with a closing-issue link: {len(linked)} ({len(linked)/len(prs):.0%})")
print(f"Merged PRs with a link: {len(linked_merged)} ({len(linked_merged)/len(merged):.0%})")

truncated = [p for p in prs if p["commits"]["totalCount"] > 100]
print(f"PRs with more than 100 commits (truncated): {len(truncated)}")

linked_issue_nums = {i["number"] for p in linked for i in p["closingIssuesReferences"]["nodes"]}
print(f"Distinct issues that have a fixing PR: {len(linked_issue_nums)}")

True 98
issues: 2768
PRs: 2913 (merged: 1634)
PRs with a closing-issue link: 341 (12%)
Merged PRs with a link: 213 (13%)
PRs with more than 100 commits (truncated): 12
Distinct issues that have a fixing PR: 300


JSON file for PRs and Issues have parameters, such as 'mergedAt' for merged PRs or 'closingIssuesReferences.nodes' for if the PR resolved least 1 issue. These parameters are referenced in its own variable and the for loops help us enter the dictionary and find the specific parameter.

In [3]:
import re
from collections import Counter

ref = re.compile(r"#(\d+)")
mentions = [p for p in prs if ref.search((p["title"] or "") + " " + (p["body"] or ""))]
print("PRs mentioning #number in title/body:", len(mentions))
print(Counter((p["author"] or {}).get("login") for p in prs).most_common(8))

PRs mentioning #number in title/body: 933
[('davidism', 306), ('dependabot', 107), ('greyli', 76), ('dependabot-preview', 54), (None, 40), ('pre-commit-ci', 40), ('untitaker', 39), ('pgjones', 35)]


933 mentions is a lot more than 341 strict links. About 32% of PRs reference a number somewhere, versus 12% with an official closing link.

In [4]:
bots = {"dependabot", "dependabot-preview", "pre-commit-ci"}
issue_nums = {i["number"] for i in issues}
pr_nums = {p["number"] for p in prs}

closing = re.compile(r"\b(?:close[sd]?|fix(?:e[sd])?|resolve[sd]?)\s*:?\s*#(\d+)", re.I)
anyref = re.compile(r"#(\d+)")

counts = Counter()
for p in prs:
    author = (p["author"] or {}).get("login")
    if author in bots:
        continue
    text = (p["title"] or "") + "\n" + (p["body"] or "")
    official = {i["number"] for i in p["closingIssuesReferences"]["nodes"]}
    fixes = {int(n) for n in closing.findall(text)}
    mentions = {int(n) for n in anyref.findall(text)} - fixes
    for n in official:
        counts["official FIXES"] += 1
    for n in fixes - official:
        counts["inferred FIXES -> issue" if n in issue_nums else "inferred FIXES -> other"] += 1
    for n in mentions - official:
        if n in issue_nums:
            counts["MENTIONS -> issue"] += 1
        elif n in pr_nums:
            counts["MENTIONS -> PR"] += 1
        else:
            counts["MENTIONS -> unknown"] += 1

for k, v in counts.most_common():
    print(f"{k}: {v}")

official FIXES: 353
MENTIONS -> issue: 237
MENTIONS -> PR: 173
inferred FIXES -> issue: 138
inferred FIXES -> other: 11
MENTIONS -> unknown: 5


load graph (terminal)

In [ ]:
'''
01_explore.ipynbdir src\graph
python src/graph/load_graph.py
'''

run the query in neo4j

In [ ]:
'''
MATCH path = (d:Developer)-[:AUTHORED]->(c:Commit)-[:PART_OF]->(p:PullRequest)-[:FIXES]->(i:Issue)
RETURN path LIMIT 25
'''